In [1]:
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import albumentations as A
from torch.optim import AdamW
from torch.utils.data import DataLoader
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedKFold
from torch.optim.lr_scheduler import CosineAnnealingLR
from optional_train import ResnetClassifier, DfToDataset, train_epoch, eval_model

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используется: {device}")

Используется: cuda


In [3]:
train_df = pd.read_csv('data/train.csv')
y = train_df['label']

IMG_DIR = 'data/train_images/'

print(train_df['label'].value_counts())

print("\nВ процентном соотношении:")
print(train_df['label'].value_counts(normalize=True) * 100)

label
3    13158
4     2577
2     2386
1     2189
0     1087
Name: count, dtype: int64

В процентном соотношении:
label
3    61.494602
4    12.043744
2    11.151096
1    10.230406
0     5.080151
Name: proportion, dtype: float64


In [4]:
train_transforms = A.Compose([
    A.Resize(height=256, width=256),  # Здесь строго height/width
    A.RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0), p=1.0),  # А здесь строго size как кортеж
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=30, p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# Трансформации для валидационной выборки
val_transforms = A.Compose([
    A.Resize(height=224, width=224),  # Тоже height/width
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

D:\Codding\Education\CV\Cassava Leaf Disease Classification\.venv\lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [5]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_metrics = []

EPOCHS = 7
for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, y)):

    X_df_train = train_df.iloc[train_idx].reset_index(drop=True)
    X_df_val = train_df.iloc[val_idx].reset_index(drop=True)

    train_dataset = DfToDataset(X_df_train, IMG_DIR, train_transforms)
    val_dataset = DfToDataset(X_df_val, IMG_DIR, val_transforms)

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

    model = ResnetClassifier().to(device)
    optimizer = AdamW(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss().to(device)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

    best_fold_metric = 0
    for epoch in range(EPOCHS):
        train_loss = train_epoch(model, optimizer, train_loader, loss_fn, device)
        val_metric_dict = eval_model(model, val_loader, device)

        scheduler.step()

        print(f"Эпоха {epoch + 1}/{EPOCHS} | Loss: {train_loss:.4f}")
        print(f"Val Precision: {val_metric_dict['precision']:.4f} | Val Recall: {val_metric_dict['recall']:.4f} | Val F1: {val_metric_dict['f1']:.4f} | Val Accuracy: {val_metric_dict['accuracy']:.4f}")

        val_metric_f1 = val_metric_dict['f1']

        if val_metric_f1 > best_fold_metric:
            best_fold_metric = val_metric_f1
            torch.save(model.state_dict(), f'models/model_fold_{fold + 1}.pth')

    fold_metrics.append(best_fold_metric)
    print(f"Лучший результат фолда {fold + 1}: {best_fold_metric}\n")

print(f"Средний результат F1 меры по фолдам: {np.mean(fold_metrics)}")


D:\Codding\Education\CV\Cassava Leaf Disease Classification\.venv\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
D:\Codding\Education\CV\Cassava Leaf Disease Classification\.venv\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Эпоха 1/7 | Loss: 0.9313
Val Precision: 0.7097 | Val Recall: 0.4314 | Val F1: 0.4208 | Val Accuracy: 0.7107
Лучший результат фолда 1: 0.42083313487953083

Эпоха 2/7 | Loss: 0.7647
Val Precision: 0.6559 | Val Recall: 0.5721 | Val F1: 0.5995 | Val Accuracy: 0.7650
Лучший результат фолда 1: 0.5994693860850614

Эпоха 3/7 | Loss: 0.6913
Val Precision: 0.6755 | Val Recall: 0.5695 | Val F1: 0.6084 | Val Accuracy: 0.7799
Лучший результат фолда 1: 0.6084263433789999

Эпоха 4/7 | Loss: 0.6324
Val Precision: 0.6566 | Val Recall: 0.6092 | Val F1: 0.6009 | Val Accuracy: 0.7521
Лучший результат фолда 1: 0.6084263433789999

Эпоха 5/7 | Loss: 0.5674
Val Precision: 0.7005 | Val Recall: 0.6555 | Val F1: 0.6576 | Val Accuracy: 0.7862
Лучший результат фолда 1: 0.6576190675732712

Эпоха 6/7 | Loss: 0.5262
Val Precision: 0.7080 | Val Recall: 0.6755 | Val F1: 0.6802 | Val Accuracy: 0.8119
Лучший результат фолда 1: 0.6802261176935442

Эпоха 7/7 | Loss: 0.4903
Val Precision: 0.7264 | Val Recall: 0.6801 | Val F

D:\Codding\Education\CV\Cassava Leaf Disease Classification\.venv\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
D:\Codding\Education\CV\Cassava Leaf Disease Classification\.venv\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Эпоха 1/7 | Loss: 0.9250
Val Precision: 0.5178 | Val Recall: 0.4321 | Val F1: 0.4030 | Val Accuracy: 0.6778
Лучший результат фолда 2: 0.40296785449813444

Эпоха 2/7 | Loss: 0.7663
Val Precision: 0.6459 | Val Recall: 0.4642 | Val F1: 0.4909 | Val Accuracy: 0.7423
Лучший результат фолда 2: 0.49087015032422354

Эпоха 3/7 | Loss: 0.6909
Val Precision: 0.5621 | Val Recall: 0.5506 | Val F1: 0.5027 | Val Accuracy: 0.6292
Лучший результат фолда 2: 0.5027011887754002

Эпоха 4/7 | Loss: 0.6225
Val Precision: 0.6845 | Val Recall: 0.5797 | Val F1: 0.5884 | Val Accuracy: 0.7724
Лучший результат фолда 2: 0.5884307390378629

Эпоха 5/7 | Loss: 0.5761
Val Precision: 0.7219 | Val Recall: 0.6404 | Val F1: 0.6561 | Val Accuracy: 0.8199
Лучший результат фолда 2: 0.656114492631324

Эпоха 6/7 | Loss: 0.5277
Val Precision: 0.7403 | Val Recall: 0.6396 | Val F1: 0.6624 | Val Accuracy: 0.8283
Лучший результат фолда 2: 0.6624436819621938

Эпоха 7/7 | Loss: 0.5028
Val Precision: 0.7610 | Val Recall: 0.6761 | Val F

D:\Codding\Education\CV\Cassava Leaf Disease Classification\.venv\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
D:\Codding\Education\CV\Cassava Leaf Disease Classification\.venv\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Эпоха 1/7 | Loss: 0.9226
Val Precision: 0.5540 | Val Recall: 0.5272 | Val F1: 0.5176 | Val Accuracy: 0.7100
Лучший результат фолда 3: 0.5176446018030658

Эпоха 2/7 | Loss: 0.7528
Val Precision: 0.6425 | Val Recall: 0.5002 | Val F1: 0.5288 | Val Accuracy: 0.7450
Лучший результат фолда 3: 0.5288100014893978

Эпоха 3/7 | Loss: 0.6798
Val Precision: 0.6957 | Val Recall: 0.5748 | Val F1: 0.6018 | Val Accuracy: 0.7883
Лучший результат фолда 3: 0.6017855948757368

Эпоха 4/7 | Loss: 0.6246
Val Precision: 0.7201 | Val Recall: 0.5954 | Val F1: 0.5919 | Val Accuracy: 0.7927
Лучший результат фолда 3: 0.6017855948757368

Эпоха 5/7 | Loss: 0.5654
Val Precision: 0.6854 | Val Recall: 0.6713 | Val F1: 0.6758 | Val Accuracy: 0.8056
Лучший результат фолда 3: 0.6758469331100491

Эпоха 6/7 | Loss: 0.5186
Val Precision: 0.7597 | Val Recall: 0.6735 | Val F1: 0.7008 | Val Accuracy: 0.8352
Лучший результат фолда 3: 0.700751105049723

Эпоха 7/7 | Loss: 0.4834
Val Precision: 0.7354 | Val Recall: 0.6997 | Val F1:

D:\Codding\Education\CV\Cassava Leaf Disease Classification\.venv\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
D:\Codding\Education\CV\Cassava Leaf Disease Classification\.venv\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Эпоха 1/7 | Loss: 0.9015
Val Precision: 0.5388 | Val Recall: 0.4717 | Val F1: 0.4739 | Val Accuracy: 0.6885
Лучший результат фолда 4: 0.4738732413277944

Эпоха 2/7 | Loss: 0.7420
Val Precision: 0.6194 | Val Recall: 0.5887 | Val F1: 0.5996 | Val Accuracy: 0.7668
Лучший результат фолда 4: 0.5995921654680253

Эпоха 3/7 | Loss: 0.6719
Val Precision: 0.6419 | Val Recall: 0.5320 | Val F1: 0.5403 | Val Accuracy: 0.7665
Лучший результат фолда 4: 0.5995921654680253

Эпоха 4/7 | Loss: 0.6159
Val Precision: 0.6908 | Val Recall: 0.6156 | Val F1: 0.6246 | Val Accuracy: 0.7817
Лучший результат фолда 4: 0.6245899628313578

Эпоха 5/7 | Loss: 0.5596
Val Precision: 0.6873 | Val Recall: 0.6198 | Val F1: 0.6364 | Val Accuracy: 0.8014
Лучший результат фолда 4: 0.6364176045285683

Эпоха 6/7 | Loss: 0.5198
Val Precision: 0.6877 | Val Recall: 0.6590 | Val F1: 0.6690 | Val Accuracy: 0.8149
Лучший результат фолда 4: 0.6689560844198806

Эпоха 7/7 | Loss: 0.4816
Val Precision: 0.7176 | Val Recall: 0.6721 | Val F1

D:\Codding\Education\CV\Cassava Leaf Disease Classification\.venv\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
D:\Codding\Education\CV\Cassava Leaf Disease Classification\.venv\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Эпоха 1/7 | Loss: 0.9261
Val Precision: 0.6299 | Val Recall: 0.4369 | Val F1: 0.4003 | Val Accuracy: 0.6504
Лучший результат фолда 5: 0.4003160783148531

Эпоха 2/7 | Loss: 0.7545
Val Precision: 0.5778 | Val Recall: 0.5702 | Val F1: 0.5434 | Val Accuracy: 0.7090
Лучший результат фолда 5: 0.5434366011130628

Эпоха 3/7 | Loss: 0.6835
Val Precision: 0.6409 | Val Recall: 0.6110 | Val F1: 0.6005 | Val Accuracy: 0.7527
Лучший результат фолда 5: 0.600523976355572

Эпоха 4/7 | Loss: 0.6189
Val Precision: 0.6751 | Val Recall: 0.6006 | Val F1: 0.6106 | Val Accuracy: 0.7932
Лучший результат фолда 5: 0.6106475979090398

Эпоха 5/7 | Loss: 0.5625
Val Precision: 0.7168 | Val Recall: 0.6315 | Val F1: 0.6418 | Val Accuracy: 0.7983
Лучший результат фолда 5: 0.6418200534332563

Эпоха 6/7 | Loss: 0.5149
Val Precision: 0.7079 | Val Recall: 0.6869 | Val F1: 0.6919 | Val Accuracy: 0.8215
Лучший результат фолда 5: 0.6919490063214515

Эпоха 7/7 | Loss: 0.4785
Val Precision: 0.7401 | Val Recall: 0.6751 | Val F1: